<a href="https://colab.research.google.com/github/saksham419/Flyrank-Assignment-Week-1/blob/main/work/notebooks/w01_research_question_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. My Lane (or Freestyle) and Why

**Chosen Lane:** Refresh / Content Opportunity Scoring

I chose this lane because the objective is to prioritize which content pages should be reviewed first. The available dataset contains observable search and engagement signals such as impressions, clicks, CTR, content age, engagement, and trend direction. These signals can support an explainable ranking that helps humans decide where to spend limited editorial effort.

# 2. The Question

**Research Question**

Which content pages should be reviewed first for refresh based on observable search and engagement signals?

**Decision:** Prioritize pages for review.

**Unit of analysis:** One content page.

**Output:** A ranked review queue with refresh scores, reason codes, and confidence.

**Action:** Content managers review the highest-ranked pages first and decide whether to refresh, expand, improve CTR, or monitor them.

**Cost of a wrong recommendation:** Time may be wasted reviewing pages that do not need attention, while important declining pages may continue losing traffic.

**Why ML can help:** Multiple signals interact in complex ways. Machine learning can combine those signals into a better ranking than simple fixed rules while still supporting human decision-making.

In [ ]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print('Rows:', len(df))
print('Declining pages (%):', round((df['trend_direction']=='down').mean()*100,2))
print('Average impressions:', round(df['impressions_90d'].mean(),2))
print('Pages with >=500 impressions (%):', round((df['impressions_90d']>=500).mean()*100,2))


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

As seen in the initial data load, the dataset contains approximately `100,000` rows. Around `30%` of pages show a declining trend, and `40%` of pages have at least 500 impressions, indicating a sizable portion of content that could benefit from review.

In [ ]:
import pandas as pd
df = pd.read_csv('/content/content_refresh_anonymized.csv')
print('--- DataFrame Info ---')
df.info()
print('\n--- DataFrame Description ---')
df.describe()

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d  

,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,sessions_prev_30d,content_age_days,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
count,27532.000000,27532.000000,27532.000000,22301.000000,22301.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.00000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000,26612.000000
mean,158.882391,0.146954,0.485342,3107.760325,20665.277835,5200.366300,16.097333,49.942467,37.066633,35.937700,...,10.283000,256.16780,4.786533,46.098300,0.510733,16.34238,2.534520,18.212921,0.768196,-4.785969
std,1518.270825,0.285241,2.101560,1452.382598,10115.344042,16838.019547,75.076958,152.101430,107.069131,103.748185,...,42.578003,132.70793,0.790392,42.078709,3.279162,15.21679,8.310096,29.472768,7.429454,473.861780
min,0.000000,0.000000,0.000000,8.000000,40.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,0.000000,90.00000,3.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,-100.000000
25%,0.000000,0.000000,0.000000,2413.000000,15644.000000,81.000000,0.000000,2.000000,2.000000,2.000000,...,1.000000,132.00000,4.000000,20.000000,0.000000,6.20000,0.000000,0.000000,0.000000,-62.600000
50%,10.000000,0.000000,0.000000,2877.000000,19116.000000,731.000000,1.000000,8.000000,7.000000,7.000000,...,2.000000,236.00000,5.000000,20.000000,0.070000,10.80000,0.000000,5.000000,0.000000,-33.500000
75%,20.000000,0.130000,0.000000,3666.000000,24011.000000,3615.250000,7.000000,33.000000,27.000000,27.000000,...,7.000000,333.00000,5.000000,104.000000,0.290000,22.30000,1.350000,23.530000,0.000000,0.000000
max,74000.000000,1.000000,100.360000,9546.000000,111158.000000,517715.000000,4178.000000,5998.000000,4345.000000,4913.000000,...,4247.000000,564.00000,6.000000,373.000000,100.000000,245.00000,100.000000,300.000000,300.000000,44900.000000


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I can claim:

My work will provide **decision-support** by offering an **observed** and **directional** ranking of content pages for refresh. It will help content managers prioritize their efforts based on measurable search and engagement signals.

### What I cannot claim:

My work **will not provide causal proof** for content performance, nor will it "predict Google" or other search engine algorithms. The recommendations are based on observable historical data and are not guarantees of future performance. It is a tool to assist human judgment, not replace it, and the ultimate decision to refresh or not rests with the content manager.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.